In [1]:
import sys
from pathlib import Path

_PROJECT_ROOT = Path.cwd().resolve()
if _PROJECT_ROOT.name == "notebooks":
    _PROJECT_ROOT = _PROJECT_ROOT.parent.parent
elif _PROJECT_ROOT.name == "podscan":
    _PROJECT_ROOT = _PROJECT_ROOT.parent
sys.path.insert(0, str(_PROJECT_ROOT))

from google_utils.google_sheet import GoogleSheetService
from data.constants import CREDENTIALS_FILE

In [2]:
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1pPF2clctk6NxAfI5AjgUJhasy8YrT1b_9XEf6MGbJe0/edit?gid=1458675702#gid=1458675702"

In [3]:
sheet_service = GoogleSheetService(credentials_file=CREDENTIALS_FILE)
spreadsheet_id = sheet_service.extract_spreadsheet_id(SPREADSHEET_URL)
success, result = sheet_service.list_sheets(spreadsheet_id)
if success:
    print("Available sheets:", result)
else:
    print("Error:", result)

Available sheets: ['List Info', 'CREative Commercial Real Estate Show', 'Global Investors: Investing in U.S. Real Estate', 'The Wealth Elevator Podcast: Real Estate, Taxes, Investing', 'Financial Freedom with Real Estate Investing', 'RealDealChat / Lessons from Real Estate Investors', 'Commercially Speaking Podcast- Commercial Real Estate Investing That Entertains', 'Happy Work - Management &amp; bien-être au travail']


In [4]:
def filter_unique_episodes(data: list, episode_id_col: str, status_col: str) -> list:
    """
    Keep unique episode_id; when duplicates exist with Pending and Completed, prefer Completed.
    """
    if not data:
        return []
    headers = data[0]
    try:
        episode_id_idx = headers.index(episode_id_col)
        status_idx = headers.index(status_col)
    except ValueError:
        return data  # columns not found, return as-is

    result_map = {}
    for row in data[1:]:
        episode_id = row[episode_id_idx] if episode_id_idx < len(row) else ""
        if not episode_id or not str(episode_id).strip():
            continue
        status = row[status_idx] if status_idx < len(row) else ""
        if episode_id not in result_map:
            result_map[episode_id] = row
        else:
            existing_status = result_map[episode_id][status_idx] if status_idx < len(result_map[episode_id]) else ""
            if existing_status == "Pending" and status == "Completed":
                result_map[episode_id] = row

    return [headers] + list(result_map.values())


# Main loop: filter duplicate episode_ids across all sheets except List Info
success, sheet_names = sheet_service.list_sheets(spreadsheet_id)
if not success or not isinstance(sheet_names, list):
    print(f"Cannot list sheets: {sheet_names}")
else:
    sheets_to_process = [s for s in sheet_names if s != "List Info"]
    for sheet_name in sheets_to_process:
        success, data = sheet_service.get_sheet_values(spreadsheet_id, f"{sheet_name}!A:Z")
        if not success:
            print(f"[{sheet_name}] Error: {data}")
            continue
        if not data:
            print(f"[{sheet_name}] No data, skipping")
            continue

        headers = data[0]
        if "episode_id" not in headers or "analysis_status" not in headers:
            print(f"[{sheet_name}] Missing episode_id or analysis_status columns, skipping")
            continue

        rows_before = len(data) - 1
        filtered = filter_unique_episodes(data, "episode_id", "analysis_status")
        rows_after = len(filtered) - 1

        success, msg = sheet_service.clear_and_rewrite_sheet(spreadsheet_id, sheet_name, filtered)
        if success:
            print(f"[{sheet_name}] {rows_before} -> {rows_after} rows ({rows_before - rows_after} duplicates removed)")
        else:
            print(f"[{sheet_name}] Error: {msg}")

[CREative Commercial Real Estate Show] 13 -> 13 rows (0 duplicates removed)
[Global Investors: Investing in U.S. Real Estate] 211 -> 211 rows (0 duplicates removed)
[The Wealth Elevator Podcast: Real Estate, Taxes, Investing] 111 -> 111 rows (0 duplicates removed)
[Financial Freedom with Real Estate Investing] 124 -> 124 rows (0 duplicates removed)
[RealDealChat / Lessons from Real Estate Investors] 298 -> 298 rows (0 duplicates removed)
[Commercially Speaking Podcast- Commercial Real Estate Investing That Entertains] 107 -> 107 rows (0 duplicates removed)
[Happy Work - Management &amp; bien-être au travail] 1 -> 1 rows (0 duplicates removed)
